# Notebook 06: Neural-Tree Hybrids & Differentiable Trees (2020–2024)
## TabNet, NODE & End-to-End Differentiable Ensembles on RTX 2050 GPU

---

### 1. Executive Intuition & The Deep Learning Tabular Dilemma

#### The Mental Model: Why Did Neural Networks Struggle on Tabular Data?
For years, standard Multi-Layer Perceptrons (MLPs) lost consistently to XGBoost on tabular benchmarks:
1. **Dense vs. Sparse Decisions**: An MLP computes linear combinations $W x + b$ that combine all features at every layer. Decision trees, by contrast, make **sparse axis-aligned cuts** (e.g. splitting solely on `order_value_sar`).
2. **Rotational Invariance**: Neural networks treat rotated coordinate systems identically, whereas tabular data has distinct semantic column meanings that cannot be rotated without destroying their domain physics.

Between 2020 and 2024, researchers closed this gap by creating **Neural-Tree Hybrids**:
* **TabNet (Google Cloud AI)**: Builds a neural network that uses **Sparsemax attention** to select a sparse subset of features at each decision step, mimicking tree splits while remaining 100% differentiable.
* **NODE (Neural Oblivious Decision Ensembles)**: Implements CatBoost-style oblivious trees as continuous, differentiable neural network layers using **$\alpha$-Entmax**.
* **NCART & GRANDE (2024)**: Hard axis-aligned decision trees trained end-to-end via gradient descent using straight-through estimators.

```mermaid
graph LR
    Input[Bilingual Tabular Features] --> Step1[Decision Step 1: Sparsemax Attention Mask]
    Step1 --> GLU1[Feature Transformer: Gated Linear Units]
    GLU1 --> Logit1[Step 1 Logit Contribution]
    GLU1 --> Prior[Update Prior P: Penalize Reused Features]
    Prior --> Step2[Decision Step 2: New Sparsemax Attention Mask]
    Step2 --> GLU2[Feature Transformer: Gated Linear Units]
    GLU2 --> Logit2[Step 2 Logit Contribution]
    Logit1 & Logit2 --> Output[Aggregated 20-Class Softmax]
```

--- 

### 2. Deep Mathematical Derivations

#### A. Sparsemax vs. Softmax for Exact Feature Selection
Standard Softmax produces strictly positive probabilities for every single input:
$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{D} e^{z_j}} > 0 \quad \forall i$$
Because $\text{softmax}(z)_i$ can never equal zero, an MLP cannot execute true feature selection—it always pays attention to every noisy column.

**Sparsemax (Martins & Astudillo, 2016)** projects input logits $z \in \mathbb{R}^D$ onto the probability simplex $\Delta^{D-1} = \{p \in \mathbb{R}^D \mid \sum p_i = 1, p_i \ge 0\}$ using Euclidean projection:
$$\text{sparsemax}(z) = \arg\min_{p \in \Delta^{D-1}} \|p - z\|_2^2$$
The closed-form analytical solution is a thresholded ReLU:
$$\mathbf{\text{sparsemax}(z)_i = \max(0, z_i - \tau(z))}$$
where the threshold $\tau(z)$ is computed by sorting coordinates $z_{(1)} \ge z_{(2)} \ge \dots \ge z_{(D)}$ and finding the cutoff index $k(z)$:
$$k(z) = \max \left\{ k \in \{1, \dots, D\} \;\middle|\; 1 + k z_{(k)} > \sum_{j=1}^{k} z_{(j)} \right\}, \quad \tau(z) = \frac{\sum_{j=1}^{k(z)} z_{(j)} - 1}{k(z)}$$

> **Mathematical Power**: Any feature whose logit $z_i \le \tau(z)$ receives an **exact mathematical zero** ($p_i = 0$). TabNet uses Sparsemax to select only the top 3–5 most relevant e-commerce features at each decision step, ignoring all other 200+ features!

#### B. TabNet Sequential Decision Architecture
At step $i$, TabNet computes a feature mask $M[i] \in [0, 1]^D$ using the previous step's representation $a[i-1]$:
$$M[i] = \text{Sparsemax}\left( P[i-1] \cdot h_i(a[i-1]) \right)$$
where $P[i-1]$ is the cumulative usage prior preventing feature re-use:
$$P[i] = \prod_{j=1}^{i} (\gamma - M[j])$$
with relaxation factor $\gamma \ge 1.0$. The masked features $M[i] \odot x$ pass through a **Gated Linear Unit (GLU)**:
$$\text{GLU}(x) = (x W_1 + b_1) \odot \sigma(x W_2 + b_2)$$
which acts as a differentiable switch.

#### C. NODE: Continuous $\alpha$-Entmax Oblivious Trees
NODE generalizes CatBoost's discrete binary oblivious tree split $\mathbb{I}(x_j > \theta)$ into a continuous differentiable split using the $\alpha$-entmax function:
$$c_d(x) = \text{entmax}_\alpha\left( \frac{W_d x - b_d}{\tau} \right)$$
where $\tau$ is a temperature parameter that cools down during training. Leaf responses are calculated as tensor outer products:
$$p(x) = \sum_{l_1=1}^2 \dots \sum_{l_D=1}^2 \left( \prod_{d=1}^D c_{d, l_d}(x) \right) W_{\text{leaf}}(l_1, \dots, l_D)$$
This allows the entire tree forest to be trained end-to-end with backpropagation and Adam!

In [ ]:
import sys, os
cur_dir = os.path.abspath(os.getcwd())
trees_dir = os.path.abspath(os.path.join(cur_dir, '..')) if os.path.basename(cur_dir) == 'notebooks' else cur_dir
repo_root = os.path.abspath(os.path.join(trees_dir, '..'))
for p in [trees_dir, repo_root]:
    if p not in sys.path: sys.path.insert(0, p)
# Setup environment and verify PyTorch CUDA for RTX 2050
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

if root_dir not in sys.path:

from trees.utils import (
    print_hardware_summary, load_dataset, prepare_features,
    evaluate_multiclass_model, plot_confusion_matrix_20, plot_metrics_comparison,
    DEPARTMENTS_EN
)

print_hardware_summary()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch CUDA Target: {device}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load 30,000 samples
df = load_dataset(sample_rows=30_000)
X_raw, y, cat_cols, num_cols = prepare_features(df)

# Preprocess for neural networks
X_processed = X_raw.copy()
scaler = StandardScaler()
X_processed[num_cols] = scaler.fit_transform(X_raw[num_cols])

for col in cat_cols:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X_raw[col].astype(str))

# 70/10/20 Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(X_processed, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.6667, random_state=42, stratify=y_temp)

print(f"Train instances: {len(X_train):,} | Test instances: {len(X_test):,}")

### 3. Model 1: TabNet (Google Cloud AI) on RTX 2050 GPU
We train `TabNetClassifier` with micro-batch sizing ($512$) budgeted to prevent CUDA OOM on our 4GB VRAM.

In [ ]:
all_results = []

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    
    t0 = time.time()
    tabnet = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=4,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params=dict(step_size=10, gamma=0.9),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        device_name=device,
        verbose=0
    )
    
    tabnet.fit(
        X_train=X_train.values, y_train=y_train,
        eval_set=[(X_val.values, y_val)],
        eval_metric=['accuracy'],
        max_epochs=25,
        patience=7,
        batch_size=512,
        virtual_batch_size=128
    )
    tabnet_time = time.time() - t0
    
    tabnet_pred = tabnet.predict(X_test.values)
    tabnet_prob = tabnet.predict_proba(X_test.values)
    tabnet_metrics = evaluate_multiclass_model("TabNet (Google Neural Tree)", y_test, tabnet_pred, tabnet_prob, tabnet_time)
    all_results.append(tabnet_metrics)
    print("TabNet Evaluation:", tabnet_metrics)
except ImportError:
    print("pytorch_tabnet not installed; skipping TabNet benchmark.")

### 4. Inspecting TabNet Sparse Attention Masks
Visualizing the top features selected by Sparsemax at each sequential decision step.

In [ ]:
try:
    explain_matrix, masks = tabnet.explain(X_test.values[:100])
    top_indices = np.argsort(explain_matrix.mean(axis=0))[-10:]
    top_names = [X_train.columns[i] for i in top_indices]
    
    plt.figure(figsize=(10, 5))
    plt.barh(top_names, explain_matrix.mean(axis=0)[top_indices], color='mediumslateblue', edgecolor='black')
    plt.title('TabNet Attention-Weighted Feature Attribution (Top 10)', fontsize=12, fontweight='bold')
    plt.xlabel('Aggregated Sparsemax Attention Weight', fontsize=11)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Could not plot TabNet masks:", e)

### 5. Summary: The Neural vs. Tree Frontier

| Criterion | Classical GBDT (CatBoost/XGBoost) | Neural-Tree Hybrid (TabNet/NODE) |
|---|---|---|
| **Training Speed** | Fast (seconds to minutes on GPU) | Slower (requires multiple gradient epochs) |
| **Data Preprocessing** | Minimal (handles unscaled data) | Requires scaling, imputation & embeddings |
| **Multi-Modal Integration**| Difficult | **Seamless (can be backpropped with CNNs/Transformers)** |
| **Semi-Supervised Pretraining**| No | **Yes (Masked self-supervised reconstruction)** |